# YOLO11 目标检测应用案例

本案例基于 MindSpore 适配实现的 `ultralytics`，演示 YOLO11 目标检测任务的训练、评估与推理完整流程。

In [8]:
import gc
import importlib
import os
import sys
import urllib.request
from pathlib import Path

import mindspore as ms

custom_ultralytics_parent = Path("/root/mindnlp/src/mindnlp")
if str(custom_ultralytics_parent) not in sys.path:
    sys.path.insert(0, str(custom_ultralytics_parent))

for module_name in list(sys.modules):
    if module_name == "ultralytics" or module_name.startswith("ultralytics."):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()

import ultralytics
from ultralytics import YOLO

ms.set_context(mode=ms.PYNATIVE_MODE, device_target="Ascend")
package_dir = Path(ultralytics.__file__).resolve().parent
os.chdir(package_dir)
work_dir = Path.cwd() / "demo_inputs"
work_dir.mkdir(parents=True, exist_ok=True)

print("MindSpore version:", ms.__version__)
print("ultralytics package:", ultralytics.__file__)
print("package_dir:", package_dir)
print("cwd:", Path.cwd())

[WARNING] ME(42604:281473233685056,MainProcess):2026-04-24-17:13:57.568.000 [mindspore/context.py:1334] For 'context.set_context', the parameter 'device_target' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.


MindSpore version: 2.8.0
ultralytics package: /root/mindnlp/src/mindnlp/ultralytics/__init__.py
package_dir: /root/mindnlp/src/mindnlp/ultralytics
cwd: /root/mindnlp/src/mindnlp/ultralytics


## 1. 数据与测试图片准备

训练与评估默认使用 `cfg/datasets/coco128.yaml`。推理阶段优先使用数据集中的样例图片；若当前环境下没有可用测试图片，则自动下载一张示例图片。

In [9]:
data_yaml = package_dir / "cfg/datasets/coco128.yaml"
finetune_model_path = package_dir / "yolo11n.pt"
scratch_model_path = package_dir / "cfg/models/11/yolo11.yaml"

def resolve_source():
    candidates = [
        package_dir / "datasets/coco128/images/train2017",
        package_dir / "datasets/coco128/images/val2017",
    ]
    suffixes = {".jpg", ".jpeg", ".png", ".bmp"}
    for candidate in candidates:
        if candidate.is_file():
            return candidate
        if candidate.is_dir():
            files = sorted([p for p in candidate.rglob("*") if p.suffix.lower() in suffixes])
            if files:
                return files[0]

    image_path = work_dir / "detect_bus.jpg"
    if not image_path.exists():
        urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", image_path.as_posix())
        print("测试图片下载完成:", image_path)
    else:
        print("测试图片已存在:", image_path)
    return image_path

source_img = resolve_source()
print("数据配置:", data_yaml)
print("推理图片:", source_img)

数据配置: /root/mindnlp/src/mindnlp/ultralytics/cfg/datasets/coco128.yaml
推理图片: /root/mindnlp/src/mindnlp/ultralytics/datasets/coco128/images/train2017/000000000009.jpg


## 2. 模型训练

检测任务支持使用预训练权重进行微调，也支持使用模型配置文件从头开始训练。

In [13]:
# 微调
#model = YOLO(finetune_model_path.as_posix())
# 从头开始训练
model = YOLO(scratch_model_path.as_posix())

train_results = model.train(
    data=data_yaml.as_posix(),
    epochs=100,
    imgsz=640,
    batch=16,
    amp=False,
    val_interval=10,
    workers=8,
)

print("训练完成。")
print("best_fitness:", getattr(train_results, "best_fitness", None))
print("save_dir:", getattr(train_results, "save_dir", None))

[MindNLP YOLO] 检测到传入 YAML 架构文件: /root/mindnlp/src/mindnlp/ultralytics/cfg/models/11/yolo11.yaml
[MindNLP YOLO] 模式: 从头开始随机初始化训练 (跳过权重转换)。
[MindNLP YOLO] 准备启动 detect 任务的训练...
[INFO] 训练任务启动，总轮数: 2 epochs
Epoch [0/1] Step [0/8] | Total Loss: 220.9965 | Box: 3.0431 | Cls: 213.7144 | DFL: 4.2390
Epoch [1/1] Step [0/8] | Total Loss: 185.0031 | Box: 3.2246 | Cls: 177.4669 | DFL: 4.3116

[INFO] 开始执行 Epoch 1 验证程序...


Validating: 100%|██████████| 8/8 [01:58<00:00, 14.77s/it]
2026-04-24 17:33:08,271 - INFO - 推理测速: preprocess: 0.0ms | inference: 471.4ms | postprocess: 404.2ms


--------------------------------------------------
[评估报告] Epoch 1
  - mAP50(B)        : 0.00000
  - mAP50-95(B)     : 0.00000
[INFO] 当前模型综合评价指标 (Fitness): 0.00000
--------------------------------------------------

[INFO] 已更新最佳模型权重 (best.ckpt)，当前最高精度: 0.0000
训练完成。
best_fitness: 2.581491654308886e-07
save_dir: runs/detect/train


## 3. 模型评估

训练完成后，在验证集上执行目标检测评估。

In [14]:
val_results = model.val(data=data_yaml.as_posix())
print("评估完成。")
print(val_results)

[MindNLP YOLO] 准备启动 detect 任务的验证...


Validating: 100%|██████████| 8/8 [01:10<00:00,  8.75s/it]
2026-04-24 17:35:36,582 - INFO - 推理测速: preprocess: 0.0ms | inference: 297.1ms | postprocess: 220.2ms


评估完成。
{'speed': {'preprocess': 0.015323981642723083, 'inference': 297.07514308393, 'postprocess': 220.2438898384571}, 'metrics/mAP50(B)': 1.430186148306948e-06, 'metrics/mAP50-95(B)': 1.430186148306948e-07, 'fitness': 1.430186148306948e-07}


## 4. 模型推理

推理结果会自动保存到输出目录，用户可直接查看生成的可视化图片。

In [15]:
gc.collect()

predict_results = model(
    source=source_img.as_posix(),
    imgsz=640,
    conf=0.25,
    iou=0.45,
    save=True,
)

print("推理完成。")
if len(predict_results) > 0:
    print("推理结果保存目录:", getattr(predict_results[0], "save_dir", None))

2026-04-24 17:35:42,424 - INFO -  推理结果将保存至: /root/mindnlp/src/mindnlp/ultralytics/runs/detect/predict
2026-04-24 17:35:42,425 - INFO - 推理引擎启动，共探测到 1 份输入样本。


[MindNLP YOLO] 准备启动 detect 任务的推理...


2026-04-24 17:35:43,465 - INFO - 处理完成 [000000000009.jpg] | 前向推理: 576.2ms | 后处理: 451.8ms


推理完成。
推理结果保存目录: /root/mindnlp/src/mindnlp/ultralytics/runs/detect/predict
